# ETL Global Superstore -> Data Warehouse (Star Schema)
**Sumber**: `Global_Superstore_250.csv` (Kaggle: `apoorvaappz/global-super-store-dataset`)  
**Output**: `DWH_Global_Superstore.xlsx`  
**Sheet**: `Staging`, `Fact_Sales`, `Dim_Date`, `Dim_Customer`, `Dim_Product`, `Dim_Geography`, `Dim_Shipping`, `Rekonsiliasi`

In [1]:
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

SRC = "Global_Superstore_250.csv"
OUT = "DWH_Global_Superstore.xlsx"

## 1. Extract

In [2]:
def extract(path):
    raw = pd.read_csv(path)
    staging = raw.copy()
    print(f"[EXTRACT] {raw.shape[0]} baris, {raw.shape[1]} kolom")
    return raw, staging

## 2. Transform

In [3]:
def snake(col):
    return re.sub(r"[\s\-]+", "_", col.strip()).lower()


def transform(raw):
    df = raw.copy()

    df.columns = [snake(c) for c in df.columns]

    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].str.strip()

    df["order_date"] = pd.to_datetime(df["order_date"], format="%d-%m-%Y")
    df["ship_date"] = pd.to_datetime(df["ship_date"], format="%d-%m-%Y")

    n_missing = df["postal_code"].isna().sum()
    df["postal_code"] = df["postal_code"].apply(
        lambda v: "Unknown" if pd.isna(v) else str(int(v)))
    print(f"[TRANSFORM] postal_code kosong diisi 'Unknown': {n_missing} baris")

    assert df.duplicated().sum() == 0, "ada baris duplikat"
    assert df["row_id"].is_unique, "row_id tidak unik"
    assert (df["sales"] > 0).all() and (df["quantity"] > 0).all()
    assert df["discount"].between(0, 1).all()
    assert (df["ship_date"] >= df["order_date"]).all()
    print("[TRANSFORM] validasi lolos: 0 duplikat, 0 pelanggaran aturan")

    df["delivery_days"] = (df["ship_date"] - df["order_date"]).dt.days
    df["profit_margin"] = df["profit"] / df["sales"]
    df["discount_level"] = pd.cut(
        df["discount"], bins=[-1, 0, 0.10, 0.30, 1.0],
        labels=["Tanpa diskon", "Rendah (<=10%)",
                "Sedang (10-30%)", "Tinggi (>30%)"]).astype(str)

    def make_dim(cols, key):
        d = df[cols].drop_duplicates().reset_index(drop=True)
        d.insert(0, key, d.index + 1)
        return d

    dim_customer = make_dim(
        ["customer_id", "customer_name", "segment"], "customer_key")
    dim_product = make_dim(
        ["product_id", "product_name", "category", "sub_category"],
        "product_key")
    dim_geo = make_dim(
        ["city", "state", "country", "region", "market", "postal_code"],
        "geography_key")
    dim_ship = make_dim(["ship_mode", "order_priority"], "shipping_key")

    dates = pd.Series(pd.concat([df["order_date"], df["ship_date"]]).unique())
    dates = dates.sort_values().reset_index(drop=True)
    dim_date = pd.DataFrame({
        "date_key": dates.dt.strftime("%Y%m%d").astype(int),
        "full_date": dates,
        "day": dates.dt.day,
        "month": dates.dt.month,
        "month_name": dates.dt.month_name(),
        "quarter": dates.dt.quarter,
        "year": dates.dt.year,
    })

    fact = df.copy()
    fact["order_date_key"] = fact["order_date"].dt.strftime("%Y%m%d").astype(int)
    fact["ship_date_key"] = fact["ship_date"].dt.strftime("%Y%m%d").astype(int)
    fact = fact.merge(dim_customer, on=["customer_id", "customer_name", "segment"])
    fact = fact.merge(dim_product,
                      on=["product_id", "product_name", "category", "sub_category"])
    fact = fact.merge(dim_geo,
                      on=["city", "state", "country", "region", "market", "postal_code"])
    fact = fact.merge(dim_ship, on=["ship_mode", "order_priority"])
    fact = fact.sort_values("row_id").reset_index(drop=True)

    fact = fact[["row_id", "order_id", "order_date_key", "ship_date_key",
                 "customer_key", "product_key", "geography_key", "shipping_key",
                 "sales", "quantity", "discount", "profit", "shipping_cost",
                 "delivery_days", "profit_margin", "discount_level"]]
    fact.insert(0, "fact_key", fact.index + 1)

    print(f"[TRANSFORM] Dim_Customer={len(dim_customer)}, Dim_Product={len(dim_product)}, "
          f"Dim_Geography={len(dim_geo)}, Dim_Shipping={len(dim_ship)}, Dim_Date={len(dim_date)}")
    return fact, dim_date, dim_customer, dim_product, dim_geo, dim_ship

## 3. Load & Rekonsiliasi

In [4]:
def load(staging, fact, dims, out):
    dim_date, dim_customer, dim_product, dim_geo, dim_ship = dims
    with pd.ExcelWriter(out, engine="openpyxl", datetime_format="yyyy-mm-dd") as w:
        staging.to_excel(w, sheet_name="Staging", index=False)
        fact.to_excel(w, sheet_name="Fact_Sales", index=False)
        dim_date.to_excel(w, sheet_name="Dim_Date", index=False)
        dim_customer.to_excel(w, sheet_name="Dim_Customer", index=False)
        dim_product.to_excel(w, sheet_name="Dim_Product", index=False)
        dim_geo.to_excel(w, sheet_name="Dim_Geography", index=False)
        dim_ship.to_excel(w, sheet_name="Dim_Shipping", index=False)
        pd.DataFrame().to_excel(w, sheet_name="Rekonsiliasi", index=False)
    print(f"[LOAD] disimpan ke {out}")


def format_and_reconcile(out):
    wb = load_workbook(out)
    head_fill = PatternFill("solid", start_color="1F3864")
    for ws in wb.worksheets:
        if ws.title == "Rekonsiliasi":
            continue
        for c in ws[1]:
            c.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
            c.fill = head_fill
            c.alignment = Alignment(horizontal="center")
        for col in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0
                        for c in list(col)[:60])
            ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 10), 40)
        ws.freeze_panes = "A2"
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = Font(name="Arial", size=10)

    ws = wb["Fact_Sales"]
    hdr = {c.value: c.column_letter for c in ws[1]}
    for name, fmt in [("sales", "#,##0.00"), ("profit", "#,##0.00"),
                      ("shipping_cost", "#,##0.00"), ("profit_margin", "0.0%"),
                      ("discount", "0%")]:
        for c in ws[hdr[name]][1:]:
            c.number_format = fmt

    r = wb["Rekonsiliasi"]
    r["A1"] = "Rekonsiliasi Sumber (Staging) vs Data Warehouse (Fact_Sales)"
    r["A1"].font = Font(name="Arial", bold=True, size=12)
    heads = ["Pemeriksaan", "Sumber (Staging)", "Data warehouse", "Selisih", "Status"]
    for i, h in enumerate(heads, 1):
        c = r.cell(row=3, column=i, value=h)
        c.font = Font(name="Arial", bold=True, color="FFFFFF")
        c.fill = head_fill
    st = wb["Staging"]; ft = wb["Fact_Sales"]
    s_hdr = {c.value: c.column_letter for c in st[1]}
    f_hdr = {c.value: c.column_letter for c in ft[1]}
    rows = [
        ("Jumlah baris",
         f"=COUNTA(Staging!{s_hdr['Row ID']}2:{s_hdr['Row ID']}1000)",
         f"=COUNTA(Fact_Sales!{f_hdr['row_id']}2:{f_hdr['row_id']}1000)"),
        ("Total Sales",
         f"=SUM(Staging!{s_hdr['Sales']}2:{s_hdr['Sales']}1000)",
         f"=SUM(Fact_Sales!{f_hdr['sales']}2:{f_hdr['sales']}1000)"),
        ("Total Profit",
         f"=SUM(Staging!{s_hdr['Profit']}2:{s_hdr['Profit']}1000)",
         f"=SUM(Fact_Sales!{f_hdr['profit']}2:{f_hdr['profit']}1000)"),
        ("Foreign key kosong pada fakta", 0,
         "=COUNTBLANK(Fact_Sales!D2:I251)"),
    ]
    for i, (name, a, b) in enumerate(rows, start=4):
        r.cell(row=i, column=1, value=name)
        r.cell(row=i, column=2, value=a)
        r.cell(row=i, column=3, value=b)
        r.cell(row=i, column=4, value=f"=C{i}-B{i}")
        r.cell(row=i, column=5, value=f'=IF(ABS(D{i})<0.01,"Sesuai","Tidak sesuai")')
        for col in range(1, 6):
            r.cell(row=i, column=col).font = Font(name="Arial", size=10)
        for col in (2, 3, 4):
            r.cell(row=i, column=col).number_format = "#,##0.00" if i in (5, 6) else "#,##0"
    r["A9"] = "Catatan: kolom D2:I251 pada Fact_Sales = 2 kunci tanggal + 4 kunci dimensi."
    r["A9"].font = Font(name="Arial", italic=True, size=9)
    for col, w in zip("ABCDE", (34, 20, 20, 14, 16)):
        r.column_dimensions[col].width = w
    wb.save(out)

## 4. Eksekusi Pipeline ETL

In [5]:
raw, staging = extract(SRC)
fact, *dims = transform(raw)
load(staging, fact, dims, OUT)
format_and_reconcile(OUT)
print("ETL selesai.")

[EXTRACT] 250 baris, 24 kolom
[TRANSFORM] postal_code kosong diisi 'Unknown': 190 baris
[TRANSFORM] validasi lolos: 0 duplikat, 0 pelanggaran aturan
[TRANSFORM] Dim_Customer=217, Dim_Product=212, Dim_Geography=199, Dim_Shipping=12, Dim_Date=379
[LOAD] disimpan ke DWH_Global_Superstore.xlsx
ETL selesai.
